# Preprocessing - Basic Train/Test Split
Simple concatenation and split

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import joblib

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

# Basic split — combine 2024+2025 as training, 2026 as test
df_train = pd.concat([df24, df25], ignore_index=True)
df_test  = df26.copy()

print(f"Train shape: {df_train.shape}")
print(f"Test  shape: {df_test.shape}")


Train shape: (141, 44)
Test  shape: (71, 44)


In [2]:

TARGET = "Target_Days"

# Selecdgst only numeric columns for now
num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]

X_train = df_train[num_cols].copy()
y_train = df_train[TARGET].astype(float).values
X_test  = df_test[num_cols].copy()
y_test  = df_test[TARGET].astype(float).values

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")
print(f"\nFeatures used ({len(num_cols)}):")
print(num_cols)


X_train: (141, 40)  y_train: (141,)
X_test : (71, 40)   y_test : (71,)

Features used (40):
['Year', 'Annual_Rounds', 'Months_In_Season', 'Near_Pruning_Flag', 'Extent_Hect', 'Age_Months', 'Days_Since_Last_Pruning', 'Prune_Cycle_Stage', 'Soil_pH', 'Soil_Carbon', 'Ph_Deviation', 'Type_VP', 'Type_SD', 'Yield_Prev_Year', 'Yield_Avg_Last3', 'Yield_Trend_Last3', 'Field_Productivity', 'Rainfall_Current', 'WetDays_Current', 'Rainfall_Lag1', 'WetDays_Lag1', 'Rainfall_Lag2', 'WetDays_Lag2', 'Rainfall_Lag3', 'WetDays_Lag3', 'Rainfall_Sum_3', 'Rain_Intensity_Lag1', 'Leaching_Risk', 'Growth_Response', 'Div_LVO', 'Div_UVO', 'Div_UDK', 'Div_LDK', 'Div_AGO', 'Target_Lag1', 'Target_Lag2', 'Solar_Current', 'Solar_Lag1', 'Solar_Lag2', 'Solar_Rain_Int']


In [3]:
# Basic imputation — fill NaN with column mean (simple, not ideal)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

print("NaN after imputation:")
print(f"  X_train: {np.isnan(X_train_imp).sum()}")
print(f"  X_test : {np.isnan(X_test_imp).sum()}")


NaN after imputation:
  X_train: 0
  X_test : 0


In [4]:
print("preprocessing summary:")
print(f"Train rows: {len(y_train)}")
print(f"Test  rows: {len(y_test)}")
print(f"Features  : {X_train_imp.shape[1]}")
print(f"Target range: {y_train.min():.2f} — {y_train.max():.2f} days")



preprocessing summary:
Train rows: 141
Test  rows: 71
Features  : 39
Target range: 9.73 — 30.00 days


Basic Split Only
- Train = 2024 + 2025 (141 rows), Test = 2026 (71 rows)
- No leakage fix 
